## Standardising Fields and Labels

Inconsistent data values are a silent killer of analysis. If the same concept is recorded in different ways, your `GROUP BY` will split it into separate categories, your `JOIN` will fail to match, and your counts will be wrong.

### Common problems

| Problem | Example | Impact |
| --- | --- | --- |
| **Case differences** | `'London'` vs `'london'` vs `'LONDON'` | Three groups instead of one |
| **Whitespace** | `' London'` vs `'London '` vs `'London'` | Joins silently fail |
| **Abbreviations** | `'Acad'` vs `'Academy'` | Incomplete grouping |
| **Encoding** | `'St Mary’s'` vs `'St Mary's'` | String comparison fails |
| **Null vs empty string** | `NULL` vs `''` vs `'N/A'` vs `'Unknown'` | Inconsistent null handling |
| **Date formats** | `'15/03/2024'` vs `'2024-03-15'` vs `'March 15, 2024'` | Type errors or wrong sorting |

### Standardisation techniques

* **`LOWER()` or `UPPER()`** — normalise case
* **`TRIM()`** — remove leading/trailing whitespace
* **`REPLACE()` / `REGEXP_REPLACE()`** — fix known abbreviations or encoding issues
* **`COALESCE()`** — unify nulls and empty strings
* **`CASE WHEN` / lookup tables** — map variant labels to a canonical form
* **`TO_DATE()` / `DATE_FORMAT()`** — parse and standardise dates

In [0]:
%sql
-- Messy data with inconsistent labels
CREATE OR REPLACE TEMP VIEW messy_schools AS
SELECT * FROM VALUES
  (100, 'Oak Academy',    'LONDON',     'Academy'),
  (101, 'oak academy',    'London',     'academy'),
  (200, ' Elm School ',   'manchester', 'Maintained'),
  (201, 'Elm School',     'Manchester', 'maintained'),
  (300, 'Birch Coll.',    'Birmingham', 'Acad'),
  (301, 'Birch College',  'birmingham', NULL)
AS t(school_urn, school_name, city, school_type);

SELECT * FROM messy_schools;

In [0]:
%sql
-- Standardise the messy data
SELECT
  school_urn,

  -- Standardise school name: trim whitespace, title case via initcap
  INITCAP(TRIM(school_name))                           AS school_name_clean,

  -- Standardise city: trim and title case
  INITCAP(TRIM(LOWER(city)))                           AS city_clean,

  -- Standardise school type: map variants to canonical labels
  CASE
    WHEN LOWER(TRIM(COALESCE(school_type, ''))) IN ('academy', 'acad')
      THEN 'Academy'
    WHEN LOWER(TRIM(COALESCE(school_type, ''))) IN ('maintained')
      THEN 'Maintained'
    ELSE 'Unknown'
  END                                                   AS school_type_clean

FROM messy_schools;

In [0]:
%sql
-- Using a lookup table for more maintainable standardisation
CREATE OR REPLACE TEMP VIEW school_type_lookup AS
SELECT * FROM VALUES
  ('academy',    'Academy'),
  ('acad',       'Academy'),
  ('maintained', 'Maintained'),
  ('la maintained', 'Maintained')
AS t(raw_value, canonical_value);

-- Join to the lookup to standardise
SELECT
  m.school_urn,
  INITCAP(TRIM(m.school_name))                         AS school_name_clean,
  INITCAP(TRIM(LOWER(m.city)))                         AS city_clean,
  COALESCE(lu.canonical_value, 'Unknown')              AS school_type_clean
FROM messy_schools m
LEFT JOIN school_type_lookup lu
  ON LOWER(TRIM(COALESCE(m.school_type, ''))) = lu.raw_value;